This file is part of MOFTy.

MOFTy is free software: you can redistribute it and/or modify it under the
terms of the GNU General Public License version 3 as published by the Free
Software Foundation.

MOFTy is distributed in the hope that it will be useful, but WITHOUT ANY
WARRANTY; without even the implied warranty of MERCHANTABILITY or FITNESS FOR
A PARTICULAR PURPOSE. See the GNU General Public License for more details.

You should have received a copy of the GNU General Public License along with
MOFTy. If not, see http://www.gnu.org/licenses/

Copyright(C) 2026 Maximilian Neumann

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

In [ ]:
import textwrap
from pathlib import Path
from scipy import sparse
from scipy.stats import kurtosis, normaltest, pearsonr

import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.cm import ScalarMappable
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from sklearn.metrics import r2_score

import gseapy as gp
import mofax
import muon as mu
import scanpy as sc
import squidpy as sq

In [ ]:
def make_puor_rdgy_cmap(n=256):
    """
    Custom colormap: Purple (low) → White (mid) → Red (high).
    Zines PuOr (purple side) and RdGy (red side).
    """
    m1 = plt.get_cmap("Purples")
    m2 = plt.get_cmap("gray_r")
    
    colors = np.zeros((n, 4))
    
    for i in range(n):
        frac = i / (n - 1)  # 0 to 1
        
        if frac > 0.5:
            # Upper half: white to purple (0 to 1 in Purples)
            colors[i] = m1((frac - 0.5) * 2)
        else:
            # Lower half: gray to white (1 to 0 in Greys, reversed)
            colors[i] = m2(1 - frac * 2)
    
    
    return mcolors.LinearSegmentedColormap.from_list("custom_cm", colors)

# Register as global colormap (optional)
if "custom_cm" in mpl.colormaps:
    mpl.colormaps.unregister("custom_cm")
mpl.colormaps.register(cmap=make_puor_rdgy_cmap())

In [ ]:
mefisto_output_dir = Path("../output/output_gbm_mefisto")
Z_model_file = mefisto_output_dir / "mefisto_model_trained_fullGP.hdf5"
interpolated_model_file = mefisto_output_dir / "mefisto_model_trained_fullGP_int.hdf5"
diff_model_file = mefisto_output_dir / "mefisto_model_trained_fullGP_diff.hdf5"

# Close handles from a previous run of this cell before overwriting files
for _name in (
    "Z_model",
    "interpolated_model",
    "diff_model",
    "mefisto",
    "mefisto_int",
    "mefisto_diff",
):
    _m = globals().get(_name)
    if _m is not None:
        _m.close()
        del globals()[_name]

interpolated_model_file.unlink(missing_ok=True)
diff_model_file.unlink(missing_ok=True)

shutil.copy(Z_model_file, interpolated_model_file)
shutil.copy(Z_model_file, diff_model_file)

mefisto = mofax.mofa_model(Z_model_file, mode="r")
mefisto_int = mofax.mofa_model(interpolated_model_file, mode="r+")
mefisto_diff = mofax.mofa_model(diff_model_file, mode="r+")

print(mefisto.training_stats["scales"])

interpolated_factors = pd.read_csv(mefisto_output_dir / "Z_interpolated_fullGP.csv", index_col=0)
interpolated_factors.columns = [f"Factor{i+1}" for i in range(interpolated_factors.shape[1])]
print(interpolated_factors)

Z_factors = np.asarray(mefisto.expectations["Z"]["group1"][:], dtype=float)
mefisto_int.expectations["Z"]["group1"][:] = interpolated_factors.to_numpy().T.astype(float)
mefisto_diff.expectations["Z"]["group1"][:] = (
    Z_factors - interpolated_factors.to_numpy().T.astype(float)
)
mefisto_int.model.flush()
mefisto_diff.model.flush()

In [ ]:
# Prepare factor metadata (plot moved to the "Plot factors" section)
h5 = mefisto.model
n_samples = mefisto.expectations["Z"]["group1"].shape[1]


def read_cov(key):
    grp = h5[key]
    if isinstance(grp, h5py.Group):
        arr = np.concatenate([np.asarray(grp[g]) for g in grp.keys()], axis=-1)
    else:
        arr = np.asarray(grp)
    arr = np.asarray(arr, dtype=float)
    if arr.shape[0] != n_samples:
        arr = arr.T
    return arr


cov = read_cov("cov_samples" if "cov_samples" in h5 else "covariates")
cov_ls = read_cov("cov_samples_transformed") if "cov_samples_transformed" in h5 else cov

ts = mefisto.model["training_stats"]
scales = np.asarray(ts["scales"][()], dtype=float).ravel()
ls_key = "length_scales" if "length_scales" in ts else "lengthscales"
length_scales = np.asarray(ts[ls_key][()], dtype=float).ravel()

D = float(np.linalg.norm(np.ptp(cov_ls[:, :2], axis=0)))
length_scales_D = length_scales / D

spacing = float(np.median(cKDTree(cov_ls[:, :2]).query(cov_ls[:, :2], k=2)[0][:, 1]))
length_scales_spacing = length_scales / spacing

factor_meta = pd.DataFrame(
    {
        "Factor": [f"F{i+1}" for i in range(len(scales))],
        "Scale": scales,
        "LengthScale_D": length_scales_D,
        "LengthScale_spots": length_scales_spacing,
    }
)
print(factor_meta)


In [ ]:
data_dir = Path("../input/input_gbm")
mdata = mu.read_h5mu(data_dir / "processed_mdata.h5mu")
output_dir = Path("../output/output_gbm_mefisto")
Z_model_file = output_dir / "mefisto_model_trained_fullGP.hdf5"
interpolated_model_file = output_dir / "mefisto_model_trained_fullGP_int.hdf5"
diff_model_file = output_dir / "mefisto_model_trained_fullGP_diff.hdf5"

m_Z = mofax.mofa_model(Z_model_file)
m_Z_int = mofax.mofa_model(interpolated_model_file)
m_Z_diff = mofax.mofa_model(diff_model_file)

mdata.obs = mdata.obs.join(m_Z.get_factors(df=True).add_suffix("_Z"))
mdata.obs = mdata.obs.join(m_Z_int.get_factors(df=True).add_suffix("_interpolated"))
mdata.obs = mdata.obs.join(m_Z_diff.get_factors(df=True).add_suffix("_diff"))

Path("../paper/gbm_mefisto").mkdir(parents=True, exist_ok=True)


# Factor correlations by model components

In [ ]:
mofax.plot_factors_correlation(m_Z)
plt.savefig(
    "../paper/gbm_mefisto/gbm_factors_correlation_Z_mefisto.pdf",
    bbox_inches="tight",
    transparent=True,
)

mofax.plot_factors_correlation(m_Z_int)
plt.savefig(
    "../paper/gbm_mefisto/gbm_factors_correlation_int_mefisto.pdf",
    bbox_inches="tight",
    transparent=True,
)

mofax.plot_factors_correlation(m_Z_diff)
plt.savefig(
    "../paper/gbm_mefisto/gbm_factors_correlation_diff_mefisto.pdf",
    bbox_inches="tight",
    transparent=True,
)


# Variance explained by factors, modalities and components

In [ ]:
def _extract_r2_series(ve):
    if isinstance(ve, dict):
        if "r2" in ve:
            data = ve["r2"]
        elif "R2" in ve:
            data = ve["R2"]
        else:
            data = ve
    else:
        data = ve

    if isinstance(data, pd.DataFrame):
        df = data.copy()
    else:
        df = pd.DataFrame(data)

    if "view" in df.columns and "View" not in df.columns:
        df = df.rename(columns={"view": "View"})
    if "r2" in df.columns and "R2" not in df.columns:
        df = df.rename(columns={"r2": "R2"})

    if "View" not in df.columns or "R2" not in df.columns:
        raise ValueError(f"Missing columns in variance explained: {list(df.columns)}")

    return df.groupby("View")["R2"].sum()


def _ensure_percent(df_or_series):
    max_val = (
        np.nanmax(df_or_series.values)
        if hasattr(df_or_series, "values")
        else np.nanmax(df_or_series)
    )
    if np.isnan(max_val):
        return df_or_series
    return df_or_series * 100 if max_val <= 1.0 else df_or_series


def _calc_per_factor_df(model, factor_indices, views):
    rows = []
    for f in factor_indices:
        ve = model.calculate_variance_explained(factors=[f])
        r2 = _extract_r2_series(ve)
        row = {"factor": f"F{f+1}"}
        for v in views:
            row[v] = r2.get(v, np.nan)
        rows.append(row)
    df = pd.DataFrame(rows).set_index("factor")
    df = df.reindex(columns=views)
    return _ensure_percent(df)


views_to_use = ["gene_exp", "protein"]
model_map = {
    r"$Z$": m_Z,
    r"$Z_{\mathrm{int}}$": m_Z_int,
    r"$Z-Z_{\mathrm{int}}$": m_Z_diff,
}
factor_order = ["F1", "F2", "F3", "F4"]

# ---- precompute per-factor and total values ----
per_factor = {}
total_r2 = {}
for name, model in model_map.items():
    per_factor[name] = _calc_per_factor_df(model, [0, 1, 2, 3], views_to_use)
    ve_total = model.calculate_variance_explained()
    total_r2[name] = _ensure_percent(_extract_r2_series(ve_total))


def _values_for_view(view):
    pf = {}
    for model_name in model_map.keys():
        pf[model_name] = per_factor[model_name][view].reindex(factor_order)
    return pf

def _truncate_cmap(cmap_name, vmin=0.5, vmax=1.0, n=256):
    base = plt.get_cmap(cmap_name)
    return mcolors.LinearSegmentedColormap.from_list(
        f"{cmap_name}_trunc", base(np.linspace(vmin, vmax, n))
    )

def _collect_all_vals():
    vals = []
    for view in views_to_use:
        pf = _values_for_view(view)
        for model_name in model_map.keys():
            vals.extend(pf[model_name].values.tolist())
    return vals


def _plot_view_grid(ax, view, norm, cmap, show_ylabels="left"):
    pf = _values_for_view(view)
    models = list(model_map.keys())
    nrows = len(factor_order)
    ncols = len(models)

    for r, factor in enumerate(factor_order):
        for c, model_name in enumerate(models):
            val = pf[model_name].loc[factor]
            color = cmap(norm(val))
            rect = patches.Rectangle(
                (c, r), 1, 1, facecolor=color, edgecolor="black", linewidth=0.25
            )
            ax.add_patch(rect)
            if not np.isnan(val):
                ax.text(
                    c + 0.5,
                    r + 0.5,
                    f"{val:.1f}",
                    ha="center",
                    va="center",
                    fontsize=5,
                    color= "white" if val > 15 else "black",
                )

    ax.set_xlim(0, ncols)
    ax.set_ylim(nrows, 0)
    ax.set_aspect("auto")
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticks([i + 0.5 for i in range(ncols)])
    ax.set_xticklabels([m for m in models], rotation=0)

    if show_ylabels == "left":
        ax.set_yticks([i + 0.5 for i in range(nrows)])
        ax.set_yticklabels(factor_order, rotation=0)
        ax.tick_params(
            axis="y", labelleft=True, left=True, labelright=False, right=False, length=0
        )
    else:
        ax.tick_params(
            axis="y",
            labelleft=False,
            left=False,
            labelright=False,
            right=False,
            length=0,
        )

    ax.tick_params(axis="both", which="major", labelsize=5, length=0, width=0.25, pad=2)

    for spine in ax.spines.values():
        spine.set_linewidth(0.25)

    # Total R2 labels above columns
    top_y = -0.02
    for c, model_name in enumerate(models):
        total_val = total_r2[model_name].get(view, np.nan)
        if not np.isnan(total_val):
            ax.text(
                c + 0.5,
                top_y,
                f"{total_val:.1f}",
                ha="center",
                va="bottom",
                fontsize=5,
                gid="total_r2",
            )

# ---- plot ----
all_vals = _collect_all_vals()
vmax = np.nanmax(all_vals) if len(all_vals) > 0 else 1.0
if np.isnan(vmax) or vmax == 0:
    vmax = 1.0

norm = mcolors.Normalize(vmin=0, vmax=20)
cmap = _truncate_cmap("custom_cm", 0.5, 1.0)

fig, axes = plt.subplots(
    1, 2, figsize=(12 / 2.54, 2.2 / 2.54), sharey=True, gridspec_kw={"wspace": 0.05}
)
_plot_view_grid(axes[0], "gene_exp", norm, cmap, show_ylabels="left")
_plot_view_grid(axes[1], "protein", norm, cmap, show_ylabels="none")

# Ensure left y-labels remain visible after shared-axis updates
axes[0].set_yticks([i + 0.5 for i in range(len(factor_order))])
axes[0].set_yticklabels(factor_order, rotation=0)
axes[0].tick_params(
    axis="y", labelleft=True, left=True, labelright=False, right=False, length=0, pad=2
)

# Subtitles below plots
axes[0].text(
    0.5, 
    -0.17, 
    "Gene expression", 
    transform=axes[0].transAxes, 
    ha="center", 
    va="top", 
    fontsize=5
)
axes[1].text(
    0.5,
    -0.17,
    "Protein expression",
    transform=axes[1].transAxes,
    ha="center",
    va="top",
    fontsize=5,
)

sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=axes, fraction=0.04, pad=0.02, shrink=0.9, aspect=15)
cbar.set_ticks([0, 10, 20])
cbar.set_ticklabels(["0", "10", "20"])
cbar.ax.tick_params(labelsize=5, length=0, width=0.25, pad=1)
cbar.outline.set_visible(False)
fig.suptitle(r"Variance explained ($R^2$ in %)", fontsize=5, y=1.12)
for ax in axes:
    for text in ax.texts:
        if text.get_gid() == "total_r2":
            continue
        try:
            val = float(text.get_text())
        except ValueError:
            continue
        text.set_color("white" if val > 7 else "black")
plt.tight_layout()
plt.savefig(
    "../paper/gbm_mefisto/gbm_variance_explained_factor_view_heatmap_mefisto.pdf",
    bbox_inches="tight",
    transparent=True,
)
plt.show()

# Plot factors

In [ ]:
mdata_gene_exp = mdata.mod["gene_exp"].copy()
sq.gr.spatial_neighbors(mdata_gene_exp)

# Row-standardize spatial connectivities (safe for isolated spots)
spatial_conn = mdata_gene_exp.obsp["spatial_connectivities"]
row_sums = np.asarray(spatial_conn.sum(axis=1)).ravel()

row_inv = np.zeros_like(row_sums, dtype=float)
np.divide(1.0, row_sums, out=row_inv, where=row_sums > 0)

row_std_conn = sparse.diags(row_inv) @ spatial_conn
mdata_gene_exp.obsp["spatial_connectivities_std"] = row_std_conn

# Create the neighbors dict structure that scanpy expects
mdata_gene_exp.uns["neighbors"] = {
    "connectivities_key": "spatial_connectivities_std",
    "distances_key": "spatial_distances",
    "params": {"n_neighbors": 6, "coord_type": "generic", "method": "umap"},
    "connectivities": row_std_conn,
    "distances": mdata_gene_exp.obsp["spatial_distances"],
}

In [ ]:
# Plot factors (columns = factors, rows = components)
factors_to_plot = 4
component_specs = [
    (r"$Z$", "Z"),
    (r"$Z_{\mathrm{int}}$", "interpolated"),
    (r"$Z-Z_{\mathrm{int}}$", "diff"),
]

color_map_key = "seismic"
image_key = None

# Resolve models from renamed variables (fallback to legacy aliases)
Z_model_for_plot = globals().get("m_Z", globals().get("m_Z"))
interpolated_model_for_plot = globals().get("m_Z_int", globals().get("m_Z_int"))
diff_model_for_plot = globals().get("m_Z_diff", globals().get("m_Z_diff"))

factors_df_map = {
    "Z": Z_model_for_plot.get_factors(df=True),
    "interpolated": interpolated_model_for_plot.get_factors(df=True),
    "diff": diff_model_for_plot.get_factors(df=True),
}

# Drop factor columns from prior runs and join current ones
suffixes_to_clear = ("_Z", "_interpolated", "_diff", "_Z_interpolated", "_Z_diff")
cols_to_drop = [
    c
    for c in mdata_gene_exp.obs.columns
    if c.startswith("Factor") and any(c.endswith(sfx) for sfx in suffixes_to_clear)
]
mdata_gene_exp.obs = mdata_gene_exp.obs.drop(columns=cols_to_drop, errors="ignore")

for _, suffix in component_specs:
    mdata_gene_exp.obs = mdata_gene_exp.obs.join(
        factors_df_map[suffix].add_suffix(f"_{suffix}"),
        how="left",
    )

# Scale and lengthscale metadata for titles
h5 = Z_model_for_plot.model
ts = h5["training_stats"]
scales = np.asarray(ts["scales"][()], dtype=float).ravel()
ls_key = "length_scales" if "length_scales" in ts else "lengthscales"
length_scales = np.asarray(ts[ls_key][()], dtype=float).ravel()

n_samples = Z_model_for_plot.expectations["Z"]["group1"].shape[1]


def read_cov(key):
    grp = h5[key]
    if isinstance(grp, h5py.Group):
        arr = np.concatenate([np.asarray(grp[g]) for g in grp.keys()], axis=-1)
    else:
        arr = np.asarray(grp)
    arr = np.asarray(arr, dtype=float)
    if arr.shape[0] != n_samples:
        arr = arr.T
    return arr


cov = read_cov("cov_samples" if "cov_samples" in h5 else "covariates")
cov_ls = read_cov("cov_samples_transformed") if "cov_samples_transformed" in h5 else cov

D = float(np.linalg.norm(np.ptp(cov_ls[:, :2], axis=0)))
length_scales_D = length_scales / D

spacing = float(np.median(cKDTree(cov_ls[:, :2]).query(cov_ls[:, :2], k=2)[0][:, 1]))
length_scales_spacing = length_scales / spacing

# Build plot grid: rows=components, cols=factors
nrows = len(component_specs)
ncols = factors_to_plot
fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(11 / 2.54, 11 / 2.54),
)
axes = np.atleast_2d(axes)

# Normalize each factor using the max absolute value across all components
for col_idx, factor_number in enumerate(range(1, factors_to_plot + 1)):
    factor_cols = [f"Factor{factor_number}_{suffix}" for _, suffix in component_specs]
    factor_max_abs = max(
        np.nanmax(np.abs(mdata_gene_exp.obs[col].to_numpy(dtype=float))) for col in factor_cols
    )
    if not np.isfinite(factor_max_abs) or factor_max_abs == 0:
        factor_max_abs = 1.0

    for row_idx, (component_label, suffix) in enumerate(component_specs):
        ax = axes[row_idx, col_idx]
        factor_key = f"Factor{factor_number}_{suffix}"
        tmp_key = f"_tmp_factor_{factor_number}_{suffix}"

        vals = mdata_gene_exp.obs[factor_key].to_numpy(dtype=float) / factor_max_abs
        mdata_gene_exp.obs[tmp_key] = vals

        moran_I = sc.metrics.morans_i(mdata_gene_exp, vals=vals)

        sc.pl.spatial(
            mdata_gene_exp,
            img_key=image_key,
            color=tmp_key,
            color_map=color_map_key,
            ax=ax,
            show=False,
            legend_loc=None,
            title="",
            size=1.5,
            vmin=-1,
            vmax=1,
            colorbar_loc=None,
        )

        ax.text(
            0.05,
            0.97,
            f"$I$: {moran_I:.2f}",
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=6,
        )

        if row_idx == 0:
            ax.set_title(
                f"$S$: {scales[col_idx]:.2f}\n"
                f"$L$: {length_scales_D[col_idx]:.2f} D, {length_scales_spacing[col_idx]:.2f} SP",
                fontsize=6,
                pad=3,
            )
        if row_idx == nrows - 1:
            ax.set_xlabel(
                f"Factor {factor_number}",
                fontsize=6,
                labelpad=2,
            )
        else:
            ax.set_xlabel("")
        # Keep only component labels on first column; remove default spatial2 elsewhere
        if col_idx == 0:
            ax.set_ylabel(component_label, fontsize=6, labelpad=1)
        else:
            ax.set_ylabel("")

        # ax.set_xlabel("")
        ax.set_xticks([])
        ax.set_yticks([])

        ylim = ax.get_ylim()
        xlim = ax.get_xlim()
        x_margin = (xlim[1] - xlim[0]) * 0.05
        y_margin = (ylim[1] - ylim[0]) * 0.1
        y_margin2 = (ylim[1] - ylim[0]) * 0.05
        ax.set_ylim(ylim[0] - y_margin2, ylim[1] + y_margin)
        ax.set_xlim(xlim[0] - x_margin, xlim[1] + x_margin)

        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(0.25)

# Cleanup temporary plotting columns
tmp_cols = [c for c in mdata_gene_exp.obs.columns if c.startswith("_tmp_factor_")]
mdata_gene_exp.obs = mdata_gene_exp.obs.drop(columns=tmp_cols, errors="ignore")

fig.subplots_adjust(left=0.07, right=0.98, bottom=0.08, top=0.93, wspace=0, hspace=-0.05)
plt.savefig(
    "../paper/gbm_mefisto/gbm_factors_mefisto_mefisto.pdf",
    bbox_inches="tight",
    transparent=True,
)
plt.show()


# Weight analysis

In [ ]:
# Get the number of factors from the model
n_factors = 4
n_to_annotate = 10

# Create a figure and a grid of subplots
# Adjust grid if you have more than 4 factors
nrows = (n_factors + 1) // 2
ncols = 2
fig, axes = plt.subplots(nrows, ncols, figsize=(14 / 2.54, 8 / 2.54), sharey=True)
axes = axes.flatten()

# Get all weights for the "gene_exp" view
weights_df = m_Z.get_weights(views="gene_exp", df=True)

weights_long = weights_df.reset_index().melt(
    id_vars=weights_df.index.name or "index",
    var_name="factor",
    value_name="value",
 )
if weights_df.index.name is None:
    weights_long = weights_long.rename(columns={"index": "feature"})

weights_long["abs_value"] = weights_long["value"].abs()
factor_stats = (
    weights_long.groupby("factor")["abs_value"].agg(
        median_abs="median",
        mad_factor=lambda x: np.median(np.abs(x - np.median(x))),
    )
    .reset_index()
 )
weights_long = weights_long.merge(factor_stats, on="factor", how="left")
weights_long["abs_dev"] = (weights_long["abs_value"] - weights_long["median_abs"]).abs()
weights_long["over_mad"] = np.where(
    weights_long["mad_factor"] > 0,
    weights_long["abs_dev"] / weights_long["mad_factor"],
    np.where(weights_long["abs_dev"] > 0, np.inf, np.nan),
 )
weights_long = weights_long[
    [
        "feature", 
        "factor", 
        "value", 
        "abs_value", 
        "median_abs", 
        "abs_dev", 
        "mad_factor", 
        "over_mad"
    ]
]

kurtosis_rows = []
for i in range(n_factors):
    ax = axes[i]
    factor_name = f"Factor{i+1}"

    if factor_name in weights_df.columns:
        # --- Factor-wise Normalization ---
        factor_weights = weights_df[factor_name]

        # Calculate kurtosis for the original weights
        kurt_value = kurtosis(factor_weights.values, fisher=False)
        p_value = normaltest(factor_weights.values).pvalue if len(factor_weights) >= 8 else np.nan
        kurtosis_rows.append(
            {"factor": factor_name, "kurtosis": kurt_value, "dagostino_pvalue": p_value}
        )

        max_abs_weight = factor_weights.abs().max()
        # Avoid division by zero if all weights are zero
        if max_abs_weight > 0:
            normalized_weights = factor_weights / max_abs_weight
        else:
            normalized_weights = factor_weights

        # Sort the normalized weights for the rank plot
        sorted_weights = normalized_weights.sort_values(ascending=False)

        # Create the scatter plot of weights vs. their rank
        ax.scatter(
            range(len(sorted_weights)),
            sorted_weights.values,
            s=1,
            alpha=1,
            color="#93cffb",
        )

        # Add a horizontal line at y=0 for reference
        ax.axhline(0, color="black", linestyle="--", linewidth=0.25)

        # --- Annotation with adjust_text ---
        texts = []

        # --- Collect top weights by absolute value ---
        top_abs_features = normalized_weights.abs().nlargest(n_to_annotate).index

        # Create a mapping from feature name to its rank for plotting
        rank_map = pd.Series(range(len(sorted_weights)), index=sorted_weights.index)

        # Collect text objects for annotation

        ax.text(
            0.95,
            0.95,
            f"F{i+1}",
            fontsize=5,
            ha="right",
            va="top",
            transform=ax.transAxes,
        )
        # Set titles and labels
        ax.set_title(f"")

        # Cleaned up axis label and tick handling
        ax.set_xlabel("")
        ax.set_ylabel("")

        ax.grid(axis="y", linestyle=":", alpha=0.6, linewidth=0.25)
        ax.set_ylim(-1.2, 1.2)  # Set y-axis limits for normalized data
        ax.tick_params(
            axis="both", which="major", length=2, width=0.25, pad=1, labelsize=5
        )

        # Conditionally hide x-tick labels for the top row
        if i < 2:  # For plots in the top row (i=0, 1)
            ax.tick_params(axis="x", labelbottom=False)

        ax.set_yticks([-1, -0.5, 0, 0.5, 1])
        # Set axis spine thickness
        for spine in ax.spines.values():
            spine.set_linewidth(0.25)

        # --- Add table with top 10 features ---
        top_features_for_table = normalized_weights.abs().nlargest(10).index
        top_weights_for_table = normalized_weights.loc[top_features_for_table]

        table_data = [
            [name, f"{weight:.2f}"] for name, weight in top_weights_for_table.items()
        ]

        table_ax = inset_axes(
            ax,
            width="50%",
            height="100%",
            bbox_to_anchor=(1.0, 0.0, 1, 1),
            bbox_transform=ax.transAxes,
            loc="lower left",
            borderpad=0,
        )
        table_ax.axis("off")

        table = table_ax.table(
            cellText=table_data,
            colWidths=[0.6, 0.4],
            cellLoc="left",
            loc="center",
            bbox=[0, 0, 1, 1],
        )

        table.auto_set_font_size(False)
        table.set_fontsize(5)
        for key, cell in table.get_celld().items():
            cell.set_edgecolor("black")
            cell.set_linewidth(0.25)
            if key[1] == 0:
                cell.set_text_props(ha="left", va="center", fontstyle="italic")
            else:
                cell.set_text_props(ha="left", va="center")
    else:
        ax.text(
            0.5, 0.5, f"{factor_name}\nnot found", ha="center", va="center", fontsize=5
        )
        ax.axis("off")

# Hide any unused subplots
for i in range(n_factors, len(axes)):
    axes[i].axis("off")

fig.subplots_adjust(left=0.08, right=0.78, bottom=0.1, top=0.9, hspace=0.1, wspace=0.55)
plt.savefig(
    "../paper/gbm_mefisto/gbm_weight_distributions_gene_exp_mefisto.pdf",
    bbox_inches="tight",
    transparent=True,
 )

kurtosis_df = pd.DataFrame(kurtosis_rows)
kurtosis_df.to_csv(
    "../paper/gbm_mefisto/gbm_gene_exp_weight_kurtosis_mefisto.csv",
    index=False,
 )

weights_long.to_csv(
    "../paper/gbm_mefisto/weights_gene_exp_mefisto.csv",
    index=False,
 )

In [ ]:
# Get the number of factors from the model
n_factors = 4
n_to_annotate = 10

# Create a figure and a grid of subplots
# Adjust grid if you have more than 4 factors
nrows = (n_factors + 1) // 2
ncols = 2
fig, axes = plt.subplots(nrows, ncols, figsize=(14 / 2.54, 8 / 2.54), sharey=True)
axes = axes.flatten()

# Get all weights for the "protein" view
weights_df = m_Z.get_weights(views="protein", df=True)

weights_long = weights_df.reset_index().melt(
    id_vars=weights_df.index.name or "index",
    var_name="factor",
    value_name="value",
 )
if weights_df.index.name is None:
    weights_long = weights_long.rename(columns={"index": "feature"})

weights_long["abs_value"] = weights_long["value"].abs()
factor_stats = (
    weights_long.groupby("factor")["abs_value"].agg(
        median_abs="median",
        mad_factor=lambda x: np.median(np.abs(x - np.median(x))),
    )
    .reset_index()
 )
weights_long = weights_long.merge(factor_stats, on="factor", how="left")
weights_long["abs_dev"] = (weights_long["abs_value"] - weights_long["median_abs"]).abs()
weights_long["over_mad"] = np.where(
    weights_long["mad_factor"] > 0,
    weights_long["abs_dev"] / weights_long["mad_factor"],
    np.where(weights_long["abs_dev"] > 0, np.inf, np.nan),
 )
weights_long = weights_long[
    [
        "feature", 
        "factor", 
        "value", 
        "abs_value", 
        "median_abs", 
        "abs_dev", 
        "mad_factor", 
        "over_mad",
    ]
]


kurtosis_rows = []
for i in range(n_factors):
    ax = axes[i]
    factor_name = f"Factor{i+1}"

    if factor_name in weights_df.columns:
        # --- Factor-wise Normalization ---
        factor_weights = weights_df[factor_name]

        # Calculate kurtosis for the original weights
        kurt_value = kurtosis(factor_weights.values, fisher=False)
        p_value = normaltest(factor_weights.values).pvalue if len(factor_weights) >= 8 else np.nan
        kurtosis_rows.append(
            {"factor": factor_name, "kurtosis": kurt_value, "dagostino_pvalue": p_value}
        )

        max_abs_weight = factor_weights.abs().max()
        # Avoid division by zero if all weights are zero
        if max_abs_weight > 0:
            normalized_weights = factor_weights / max_abs_weight
        else:
            normalized_weights = factor_weights

        # Sort the normalized weights for the rank plot
        sorted_weights = normalized_weights.sort_values(ascending=False)

        # Create the scatter plot of weights vs. their rank
        ax.scatter(
            range(len(sorted_weights)),
            sorted_weights.values,
            s=1,
            alpha=1,
            color="#93cffb",
        )

        # Add a horizontal line at y=0 for reference
        ax.axhline(0, color="black", linestyle="--", linewidth=0.25)

        # --- Annotation with adjust_text ---
        texts = []

        # --- Collect top weights by absolute value ---
        top_abs_features = normalized_weights.abs().nlargest(n_to_annotate).index

        # Create a mapping from feature name to its rank for plotting
        rank_map = pd.Series(range(len(sorted_weights)), index=sorted_weights.index)

        # Collect text objects for annotation

        ax.text(
            0.95,
            0.95,
            f"F{i+1}",
            fontsize=5,
            ha="right",
            va="top",
            transform=ax.transAxes,
        )
        # Set titles and labels
        ax.set_title(f"")

        # Cleaned up axis label and tick handling
        ax.set_xlabel("")
        ax.set_ylabel("")

        ax.grid(axis="y", linestyle=":", alpha=0.6, linewidth=0.25)
        ax.set_ylim(-1.2, 1.2)  # Set y-axis limits for normalized data
        ax.tick_params(
            axis="both", which="major", length=2, width=0.25, pad=1, labelsize=5
        )

        # Conditionally hide x-tick labels for the top row
        if i < 2:  # For plots in the top row (i=0, 1)
            ax.tick_params(axis="x", labelbottom=False)

        ax.set_yticks([-1, -0.5, 0, 0.5, 1])
        # Set axis spine thickness
        for spine in ax.spines.values():
            spine.set_linewidth(0.25)

        # --- Add table with top 10 features ---
        top_features_for_table = normalized_weights.abs().nlargest(10).index
        top_weights_for_table = normalized_weights.loc[top_features_for_table]

        table_data = [
            [name.replace("_protein", ""), f"{weight:.2f}"] for name, weight in top_weights_for_table.items()
        ]

        table_ax = inset_axes(
            ax,
            width="50%",
            height="100%",
            bbox_to_anchor=(1.0, 0.0, 1, 1),
            bbox_transform=ax.transAxes,
            loc="lower left",
            borderpad=0,
        )
        table_ax.axis("off")

        table = table_ax.table(
            cellText=table_data,
            colWidths=[0.6, 0.4],
            cellLoc="left",
            loc="center",
            bbox=[0, 0, 1, 1],
        )

        table.auto_set_font_size(False)
        table.set_fontsize(5)
        for key, cell in table.get_celld().items():
            cell.set_edgecolor("black")
            cell.set_linewidth(0.25)
            cell.set_text_props(ha="left", va="center")
    else:
        ax.text(
            0.5, 0.5, f"{factor_name}\nnot found", ha="center", va="center", fontsize=5
        )
        ax.axis("off")

# Hide any unused subplots
for i in range(n_factors, len(axes)):
    axes[i].axis("off")

fig.subplots_adjust(left=0.08, right=0.78, bottom=0.1, top=0.9, hspace=0.1, wspace=0.55)
plt.savefig(
    "../paper/gbm_mefisto/gbm_weight_distributions_protein_mefisto.pdf",
    bbox_inches="tight",
    transparent=True,
 )

kurtosis_df = pd.DataFrame(kurtosis_rows)
kurtosis_df.to_csv(
    "../paper/gbm_mefisto/gbm_protein_weight_kurtosis_mefisto.csv",
    index=False,
 )


weights_long.to_csv(
    "../paper/gbm_mefisto/weights_protein_mefisto.csv",
    index=False,
 )

In [ ]:
def featurewise_r2_for_cols(model, view, cols, factors_df=None):
    """Feature-wise $R^2$ for a subset of factors (cols) using r2_score."""
    if factors_df is None:
        factors_df = model.get_factors(df=True)
    X_df = model.get_data(views=view, df=True)
    W_df = model.get_weights(views=view, df=True)

    # Warn on duplicates but do not modify indices
    if X_df.columns.has_duplicates:
        dup_x = X_df.columns[X_df.columns.duplicated()].unique().tolist()
        print(f"Warning: duplicate features in X ({view}):", dup_x)
    if W_df.index.has_duplicates:
        dup_w = W_df.index[W_df.index.duplicated()].unique().tolist()
        print(f"Warning: duplicate features in W ({view}):", dup_w)

    # Align samples and features
    X_df = X_df.reindex(factors_df.index)
    features = W_df.index
    X_df = X_df[features]
    W_sub = W_df.loc[features, cols]
    Z_sub = factors_df[cols]

    X = X_df.to_numpy()
    X_hat = Z_sub.to_numpy() @ W_sub.to_numpy().T

    r2_vals = []
    for idx in range(X.shape[1]):
        y_true = X[:, idx]
        y_pred = X_hat[:, idx]
        try:
            r2_vals.append(r2_score(y_true, y_pred))
        except Exception:
            r2_vals.append(np.nan)
    return pd.Series(r2_vals, index=features, name="var_explained_r2")


def weights_report_abs_max(weights_df, factor_col, r2_Z, r2_interpolated, r2_diff):
    """Plain abs-max normalization for a single factor column."""
    report = pd.DataFrame({
        "feature": weights_df.index,
        "weight": weights_df[factor_col].values,
    })
    report["abs_weight"] = report["weight"].abs()
    max_abs = report["abs_weight"].max()
    if max_abs == 0 or np.isnan(max_abs):
        report["norm_weight"] = 0.0
    else:
        report["norm_weight"] = report["weight"] / max_abs
    report["abs_norm_weight"] = report["norm_weight"].abs()
    report = report.set_index("feature")
    report = report.join(r2_Z.rename("var_explained_r2_Z"), how="left")
    report = report.join(r2_interpolated.rename("var_explained_r2_interpolated"), how="left")
    report = report.join(r2_diff.rename("var_explained_r2_diff"), how="left")
    report = report.reset_index()
    return report


 # --- Plain abs-max normalized report with minimal columns ---
factors_df = m_Z.get_factors(df=True)
factors_df_interpolated = m_Z_int.get_factors(df=True)
factors_df_diff = m_Z_diff.get_factors(df=True)
cols_list = [["Factor1"], ["Factor2"], ["Factor3"], ["Factor4"]]

for cols in cols_list:
    factor_col = cols[0]
    for view in ["gene_exp", "protein"]:
        weights = m_Z.get_weights(views=view, df=True)
        r2_feat_Z = featurewise_r2_for_cols(
            m_Z, view, cols, factors_df=factors_df
        )
        r2_feat_interpolated = featurewise_r2_for_cols(
            m_Z_int, view, cols, factors_df=factors_df_interpolated
        )
        r2_feat_diffh = featurewise_r2_for_cols(
            m_Z_diff, view, cols, factors_df=factors_df_diff
        )
        report = weights_report_abs_max(
            weights, factor_col, r2_feat_Z, r2_feat_interpolated, r2_feat_diffh
        )
        report = report[
            [
                "feature",
                "weight",
                "abs_weight",
                "norm_weight",
                "abs_norm_weight",
                "var_explained_r2_Z",
                "var_explained_r2_interpolated",
                "var_explained_r2_diff",
            ]
        ]

        out_dir = Path("../paper/gbm_mefisto/gbm_weights_mefisto")
        out_dir.mkdir(parents=True, exist_ok=True)
        factor_tag = factor_col.replace("Factor", "F")
        report.to_csv(out_dir / f"{view}_{factor_tag}_mefisto.csv", index=False)

### Visualization of top feature reconstruction with MEFISTO

In [ ]:
# --- Manual plotting grid ---
# Each row is a factor, columns are: Original, Z Rec, Interpolated Rec, Z-Interpolated Rec, MEFISTO Rec
nrows = 4
ncols = 4

# Map features to their NIFTde reconstruction file paths
sign = {
    "HBA2": "+",
    "VSNL1": "+",
    "COL3A1": "+",
    "APOC1": "+",
}

sign_pad = {
    "HBA2": 0.61,
    "VSNL1": 0.63,  
    "COL3A1": 0.65,
    "APOC1": 0.63,
}
# Create the Figure and Axes Grid with Matplotlib
fig, axes = plt.subplots(
    nrows=nrows, ncols=ncols, figsize=(1.1*(9 / 2.54), 1.1*(12 / 2.54))
)  # Adjusted size

# Get rna data from m_Z and align to mdata_gene_exp.obs
rna_data = m_Z.get_data(views="gene_exp", df=True)
rna_df = rna_data.copy()
rna_df = rna_df.reindex(mdata_gene_exp.obs.index)

full_data = rna_df.to_numpy()

def reconstruct_model_for_gene(model, feature, adata, view):
    """Reconstructs expression for a single gene from a model."""
    try:
        w = model.get_weights(df=True, views=view)
        f = model.get_factors(df=True)

        # Ensure gene exists in weights
        if feature not in w.index:
            return np.zeros(adata.n_obs)

        # Align factors and weights
        common_factors = [c for c in f.columns if c in w.columns]
        if not common_factors:
            return np.zeros(adata.n_obs)

        F = f[common_factors].to_numpy()
        # Get weights for the specific gene
        W_feature = w.loc[[feature], common_factors].to_numpy()

        # Reconstruct for the gene
        Xr_gene = (F @ W_feature.T).ravel()
        return Xr_gene
    except Exception:
        return np.zeros(adata.n_obs)

for row_idx, factor in enumerate([1, 2, 3, 4]):
    # Get the top feature for the current factor
    top_features_df = m_Z.get_top_features(
        factors=f"Factor{factor}", n_features=1, df=True
    )
    top_features_df = top_features_df[top_features_df["view"] == "gene_exp"]
    top_features = top_features_df["feature"].tolist()
    col_counter = 0
    feat = top_features[0]
    ax_orig = axes[row_idx, 0]
    data_orig = rna_df[[feat]].to_numpy()

    rec_Z = reconstruct_model_for_gene(
        m_Z, feat, mdata_gene_exp, view="gene_exp"
    )
    rec_interpolated = reconstruct_model_for_gene(
        m_Z_int, feat, mdata_gene_exp, view="gene_exp"
    )
    rec_diff = reconstruct_model_for_gene(
        m_Z_diff, feat, mdata_gene_exp, view="gene_exp"
    )

    max_val = max(
        np.max(np.abs(rec_Z)),
        np.max(np.abs(rec_interpolated)),
        np.max(np.abs(rec_diff)),
        np.max(np.abs(data_orig))
    )


    data_orig = data_orig.ravel()

    # Store in mdata_gene_exp.obs for plotting
    mdata_gene_exp.obs[f"centered_{feat}"] = data_orig
    sc.pl.spatial(
        mdata_gene_exp,
        img_key=None,
        color=f"centered_{feat}",
        size=1.5,
        show=False,
        title="",
        ax=ax_orig,
        color_map="seismic",
        legend_loc=None,
        vmin=-max_val,
        vmax=max_val,
        colorbar_loc=None,
    )  # Use vmin/vmax for consistent colors
    
    ax_orig.text(
        0.03,
        0.97,
        f"$I$: {sc.metrics.morans_i(mdata_gene_exp, vals=data_orig):.2f}",
        transform=ax_orig.transAxes,
        ha="left",
        va="top",
        fontsize=5,
    )
    if row_idx == 0:
        ax_orig.set_title("Original data", fontsize=6, pad=3)

    ax_orig.set_ylabel("")

    ax_orig.text(
        -0.10, 0.4, f"{feat}",
        transform=ax_orig.transAxes,
        rotation=90,
        va="center", ha="center",
        fontsize=6,
        fontstyle="italic",
    )

    # 3) Normaler Teil (sign)
    ax_orig.text(
        -0.10, sign_pad[feat], f"({sign[feat]})",
        transform=ax_orig.transAxes,
        rotation=90,
        va="center", ha="center",
        fontsize=6,
        fontstyle="normal",
    )

    col_counter += 1


    # Calculate R-squared for each reconstruction against the original data
    r2_Z = r2_score(data_orig, rec_Z)
    r2_interpolated = r2_score(data_orig, rec_interpolated)
    r2_diff = r2_score(data_orig, rec_diff)

    # Store in mdata_gene_exp.obs for plotting
    mdata_gene_exp.obs[f"rec_Z_{feat}"] = rec_Z
    mdata_gene_exp.obs[f"rec_interpolated_{feat}"] = rec_interpolated
    mdata_gene_exp.obs[f"rec_diff_{feat}"] = rec_diff

    # --- Z Reconstruction ---
    ax_Z = axes[row_idx, col_counter]
    sc.pl.spatial(
        mdata_gene_exp,
        img_key=None,
        color=f"rec_Z_{feat}",
        size=1.5,
        show=False,
        title="",
        ax=ax_Z,
        color_map="seismic",
        legend_loc=None,
        vmin=-max_val,
        vmax=max_val,
        colorbar_loc=None,
    )  # Use vmin/vmax for consistent colors

    ax_Z.text(
        0.03,
        0.97,
        f"$R^2$: {100*r2_Z:.1f}%, $I$: {sc.metrics.morans_i(mdata_gene_exp, vals=rec_Z):.2f}",
        transform=ax_Z.transAxes,
        ha="left",
        va="top",
        fontsize=5,
    )
    if row_idx == 0:
        ax_Z.set_title(r"$W_{\mathrm{ge}} Z$", fontsize=6, pad=3)

    # --- Interpolated Reconstruction ---
    ax_int = axes[row_idx, col_counter + 1]
    sc.pl.spatial(
        mdata_gene_exp,
        img_key=None,
        color=f"rec_interpolated_{feat}",
        size=1.5,
        show=False,
        title="",
        ax=ax_int,
        color_map="seismic",
        legend_loc=None,
        vmin=-max_val,
        vmax=max_val,
        colorbar_loc=None,
    )

    ax_int.text(
        0.03,
        0.97,
        f"$R^2$: {100*r2_interpolated:.1f}%, $I$: {sc.metrics.morans_i(mdata_gene_exp, vals=rec_interpolated):.2f}",
        transform=ax_int.transAxes,
        ha="left",
        va="top",
        fontsize=5,
    )
    if row_idx == 0:
        ax_int.set_title(r"$W_{\mathrm{ge}}\,Z_{\mathrm{int}}$", fontsize=6, pad=3)

    # --- Z-Interpolated Reconstruction ---
    ax_diff = axes[row_idx, col_counter + 2]
    sc.pl.spatial(
        mdata_gene_exp,
        img_key=None,
        color=f"rec_diff_{feat}",
        size=1.5,
        show=False,
        title="",
        ax=ax_diff,
        color_map="seismic",
        legend_loc=None,
        vmin=-max_val,
        vmax=max_val,
        colorbar_loc=None,
    )

    ax_diff.text(
        0.03,
        0.97,
        f"$R^2$: {100*r2_diff:.1f}%, $I$: {sc.metrics.morans_i(mdata_gene_exp, vals=rec_diff):.2f}",
        transform=ax_diff.transAxes,
        ha="left",
        va="top",
        fontsize=5,
    )
    if row_idx == 0:
        ax_diff.set_title(r"$W_{\mathrm{ge}}\,(Z-Z_{\mathrm{int}})$", fontsize=6, pad=3)

for ax in axes.flatten():
    if ax.get_ylabel() not in ("spatial1", "spatial2", ""):
        pass  # Keep the y-labels for factor groups
    else:
        ax.set_ylabel("")  # Clear scanpy"s default y-labels
    ax.set_xlabel("")  # Clear scanpy"s default x-labels

    # Add margins inside the plot
    ylim = ax.get_ylim()
    y_margin = (ylim[1] - ylim[0]) * 0.10
    # y_margin2 = (ylim[1] - ylim[0]) * 0.2
    ax.set_ylim(ylim[0], ylim[1] + y_margin)
    xlim = ax.get_xlim()
    x_margin = (xlim[1] - xlim[0]) * 0.0
    ax.set_xlim(xlim[0] - x_margin, xlim[1] + x_margin)

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.25)

# Adjust layout and show the plot
fig.subplots_adjust(left=0.1, right=0.9, bottom=0.1, top=0.9, wspace=0, hspace=-0.13)
plt.savefig(
    "../paper/gbm_mefisto/gbm_top_features_reconstruction_gene_exp_mefisto.pdf",
    bbox_inches="tight",
    transparent=True,
)

In [ ]:
mdata_protein = mdata.mod["protein"].copy()
sq.gr.spatial_neighbors(mdata_protein)

# Row-standardize spatial connectivities (safe for isolated spots)
spatial_conn = mdata_gene_exp.obsp["spatial_connectivities"]
row_sums = np.asarray(spatial_conn.sum(axis=1)).ravel()

row_inv = np.zeros_like(row_sums, dtype=float)
np.divide(1.0, row_sums, out=row_inv, where=row_sums > 0)

row_std_conn = sparse.diags(row_inv) @ spatial_conn
mdata_gene_exp.obsp["spatial_connectivities_std"] = row_std_conn

# Create the neighbors dict structure that scanpy expects
mdata_protein.uns["neighbors"] = {
    "connectivities_key": "spatial_connectivities_std",
    "distances_key": "spatial_distances",
    "params": {"n_neighbors": 6, "coord_type": "generic", "method": "umap"},
    "connectivities": row_std_conn,
    "distances": mdata_protein.obsp["spatial_distances"],
}

In [ ]:
# --- Manual plotting grid ---
# Each row is a factor, columns are: Original, Z Rec, Interpolated Rec, Z-Interpolated Rec, MEFISTO Rec
nrows = 4
ncols = 4

# Map features to their NIFTde reconstruction file paths
sign = {
    "PCNA_protein": "\u2212",
    "HLA-DRA": "\u2212",
    "ACTA2_protein": "+",
    "CD68_protein": "+",
}

# Create the Figure and Axes Grid with Matplotlib
fig, axes = plt.subplots(
    nrows=nrows, ncols=ncols, figsize=(1.1*(9 / 2.54), 1.1*(12 / 2.54))
)  # Adjusted size

# Get rna data from m_Z and align to mdata_protein.obs
rna_data = m_Z.get_data(views="protein", df=True)
rna_df = rna_data.copy()
rna_df = rna_df.reindex(mdata_protein.obs.index)

full_data = rna_df.to_numpy()

def reconstruct_model_for_gene(model, feature, adata, view):
    """Reconstructs expression for a single gene from a model."""
    try:
        w = model.get_weights(df=True, views=view)
        f = model.get_factors(df=True)

        # Ensure gene exists in weights
        if feature not in w.index:
            return np.zeros(adata.n_obs)

        # Align factors and weights
        common_factors = [c for c in f.columns if c in w.columns]
        if not common_factors:
            return np.zeros(adata.n_obs)

        F = f[common_factors].to_numpy()
        # Get weights for the specific gene
        W_feature = w.loc[[feature], common_factors].to_numpy()

        # Reconstruct for the gene
        Xr_gene = (F @ W_feature.T).ravel()
        return Xr_gene
    except Exception:
        return np.zeros(adata.n_obs)

for row_idx, factor in enumerate([1, 2, 3, 4]):
    # Get the top feature for the current factor
    top_features_df = m_Z.get_top_features(
        factors=f"Factor{factor}", n_features=1, df=True
    )
    top_features_df = top_features_df[top_features_df["view"] == "protein"]
    top_features = top_features_df["feature"].tolist()
    col_counter = 0
    feat = top_features[0]
    ax_orig = axes[row_idx, 0]
    data_orig = rna_df[[feat]].to_numpy()

    rec_Z = reconstruct_model_for_gene(
        m_Z, feat, mdata_protein, view="protein"
    )
    rec_interpolated = reconstruct_model_for_gene(
        m_Z_int, feat, mdata_protein, view="protein"
    )
    rec_diff = reconstruct_model_for_gene(
        m_Z_diff, feat, mdata_protein, view="protein"
    )

    max_val = max(
        np.max(np.abs(rec_Z)),
        np.max(np.abs(rec_interpolated)),
        np.max(np.abs(rec_diff)),
        np.max(np.abs(data_orig))
    )


    data_orig = data_orig.ravel()

    # Store in mdata_protein.obs for plotting
    mdata_protein.obs[f"centered_{feat}"] = data_orig
    sc.pl.spatial(
        mdata_protein,
        img_key=None,
        color=f"centered_{feat}",
        size=1.5,
        show=False,
        title="",
        ax=ax_orig,
        color_map="seismic",
        legend_loc=None,
        vmin=-max_val,
        vmax=max_val,
        colorbar_loc=None,
    )  # Use vmin/vmax for consistent colors
    
    ax_orig.text(
        0.03,
        0.97,
        f"$I$: {sc.metrics.morans_i(mdata_protein, vals=data_orig):.2f}",
        transform=ax_orig.transAxes,
        ha="left",
        va="top",
        fontsize=5,
    )
    if row_idx == 0:
        ax_orig.set_title("Original data", fontsize=6, pad=3)

    ax_orig.set_ylabel("")

    ax_orig.text(
        -0.10, 0.5, f"{feat.replace('_protein', '')} ({sign[feat]})",
        transform=ax_orig.transAxes,
        rotation=90,
        va="center", ha="center",
        fontsize=6,
        fontstyle="normal",
    )

    col_counter += 1


    # Calculate R-squared for each reconstruction against the original data
    r2_Z = r2_score(data_orig, rec_Z)
    r2_interpolated = r2_score(data_orig, rec_interpolated)
    r2_diff = r2_score(data_orig, rec_diff)

    # Store in mdata_protein.obs for plotting
    mdata_protein.obs[f"rec_Z_{feat}"] = rec_Z
    mdata_protein.obs[f"rec_interpolated_{feat}"] = rec_interpolated
    mdata_protein.obs[f"rec_diff_{feat}"] = rec_diff

    # --- Z Reconstruction ---
    ax_Z = axes[row_idx, col_counter]
    sc.pl.spatial(
        mdata_protein,
        img_key=None,
        color=f"rec_Z_{feat}",
        size=1.5,
        show=False,
        title="",
        ax=ax_Z,
        color_map="seismic",
        legend_loc=None,
        vmin=-max_val,
        vmax=max_val,
        colorbar_loc=None,
    )  # Use vmin/vmax for consistent colors

    ax_Z.text(
        0.03,
        0.97,
        f"$R^2$: {100*r2_Z:.1f}%, $I$: {sc.metrics.morans_i(mdata_protein, vals=rec_Z):.2f}",
        transform=ax_Z.transAxes,
        ha="left",
        va="top",
        fontsize=5,
    )
    if row_idx == 0:
        ax_Z.set_title(r"$W_{\mathrm{pe}}\,Z$", fontsize=6, pad=3)

    # --- Interpolated reconstruction ---
    ax_int = axes[row_idx, col_counter + 1]
    sc.pl.spatial(
        mdata_protein,
        img_key=None,
        color=f"rec_interpolated_{feat}",
        size=1.5,
        show=False,
        title="",
        ax=ax_int,
        color_map="seismic",
        legend_loc=None,
        vmin=-max_val,
        vmax=max_val,
        colorbar_loc=None,
    )

    ax_int.text(
        0.03,
        0.97,
        f"$R^2$: {100*r2_interpolated:.1f}%, $I$: {sc.metrics.morans_i(mdata_protein, vals=rec_interpolated):.2f}",
        transform=ax_int.transAxes,
        ha="left",
        va="top",
        fontsize=5,
    )
    if row_idx == 0:
        ax_int.set_title(r"$W_{\mathrm{pe}}\,Z_{\mathrm{int}}$", fontsize=6, pad=3)

    # --- Diff reconstruction ---
    ax_diff = axes[row_idx, col_counter + 2]
    sc.pl.spatial(
        mdata_protein,
        img_key=None,
        color=f"rec_diff_{feat}",
        size=1.5,
        show=False,
        title="",
        ax=ax_diff,
        color_map="seismic",
        legend_loc=None,
        vmin=-max_val,
        vmax=max_val,
        colorbar_loc=None,
    )

    ax_diff.text(
        0.03,
        0.97,
        f"$R^2$: {100*r2_diff:.1f}%, $I$: {sc.metrics.morans_i(mdata_protein, vals=rec_diff):.2f}",
        transform=ax_diff.transAxes,
        ha="left",
        va="top",
        fontsize=5,
    )
    if row_idx == 0:
        ax_diff.set_title(r"$W_{\mathrm{pe}}(Z-Z_{\mathrm{int}})$", fontsize=6, pad=3)

for ax in axes.flatten():
    if ax.get_ylabel() not in ("spatial1", "spatial2", ""):
        pass  # Keep the y-labels for factor groups
    else:
        ax.set_ylabel("")  # Clear scanpy"s default y-labels
    ax.set_xlabel("")  # Clear scanpy"s default x-labels

    # Add margins inside the plot
    ylim = ax.get_ylim()
    y_margin = (ylim[1] - ylim[0]) * 0.10
    # y_margin2 = (ylim[1] - ylim[0]) * 0.2
    ax.set_ylim(ylim[0], ylim[1] + y_margin)
    xlim = ax.get_xlim()
    x_margin = (xlim[1] - xlim[0]) * 0.0
    ax.set_xlim(xlim[0] - x_margin, xlim[1] + x_margin)

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.25)

# Adjust layout and show the plot
fig.subplots_adjust(left=0.1, right=0.9, bottom=0.1, top=0.9, wspace=0, hspace=-0.13)
plt.savefig(
    "../paper/gbm_mefisto/gbm_top_features_reconstruction_protein_mefisto.pdf",
    bbox_inches="tight",
    transparent=True,
)

### Gene set enrichment analysis

In [ ]:
def summarize_gsea_results(
    gsea_dict,
    view_label,
    gene_set_library,
    model_map,
    fdr_cutoff=0.25,
    nom_p_cutoff=0.05,
    top_n=20,
    view_for_r2="gene_exp",
):
    rows = []
    for factor, df in gsea_dict.items():
        if df is None or df.empty:
            continue
        df = df.copy()
        # Standardize column names
        df.columns = [c.upper() for c in df.columns]
        # Expected: NES, FDR Q-VAL, PVAL, TERM (or GENESET/TERMS)
        term_col = (
            "TERM"
            if "TERM" in df.columns
            else ("TERMS" if "TERMS" in df.columns else df.columns[0])
        )
        fdr_col = (
            "FDR Q-VAL"
            if "FDR Q-VAL" in df.columns
            else ("FDR" if "FDR" in df.columns else None)
        )
        nom_p_col = (
            "NOM P-VAL"
            if "NOM P-VAL" in df.columns
            else (
                "PVAL"
                if "PVAL" in df.columns
                else ("NOM P VAL" if "NOM P VAL" in df.columns else None)
            )
        )
        nes_col = "NES" if "NES" in df.columns else None

        if fdr_col is None or nom_p_col is None or nes_col is None:
            continue

        df = df.assign(_abs_nes_sort=df[nes_col].abs()).sort_values(
            by=[fdr_col, nom_p_col, "_abs_nes_sort"], ascending=[True, True, False]
        )
        df = df[(df[nom_p_col] < nom_p_cutoff) & (df[fdr_col] < fdr_cutoff)].head(top_n)
        df = df.drop(columns=["_abs_nes_sort"])

        for _, r in df.iterrows():
            term = r[term_col]
            genes = gene_set_library.get(term, [])
            r2_vals = {}
            for model_label, model in model_map.items():
                r2_vals[model_label] = _factor_var_explained_for_genes(
                    model, view_for_r2, factor, genes
                )
            rows.append(
                {
                    "view": view_label,
                    "factor": factor,
                    "term": term,
                    "NES": r[nes_col],
                    "NOM_pval": r[nom_p_col],
                    "FDR_qval": r[fdr_col],
                    "var_explained_Z": r2_vals.get("Z"),
                    "var_explained_interpolated": r2_vals.get("Interpolated"),
                    "var_explained_diff": r2_vals.get("$Z-$Interpolated"),
                }
            )
    return pd.DataFrame(rows)


def run_gsea_from_report(
    report_csv, factor_tag, out_dir, gene_set="Reactome_2022", score_col="value"
):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(report_csv, index_col=0)

    if score_col not in df.columns:
        raise ValueError(f"Missing column '{score_col}' in {report_csv}")

    if "factor" not in df.columns:
        raise ValueError(f"Missing column 'factor' in {report_csv}")

    df_factor = df[df["factor"] == factor_tag].copy()

    if df_factor.empty:
        raise ValueError(f"No rows found for factor '{factor_tag}' in {report_csv}")

    rnk = (
        df_factor.loc[df_factor[score_col].notna(), [score_col]]
        .reset_index()
        .rename(columns={df_factor.index.name or "index": "gene", score_col: "score"})
    )

    rnk["gene"] = rnk["gene"].astype(str)
    rnk["score"] = pd.to_numeric(rnk["score"], errors="coerce")
    rnk = rnk.dropna(subset=["score"])

    print(f"{factor_tag}: n rows before duplicate handling = {len(rnk)}")
    print(f"{factor_tag}: duplicate genes = {rnk['gene'].duplicated().sum()}")
    print(f"{factor_tag}: duplicate scores = {rnk['score'].duplicated().sum()}")
    print(f"{factor_tag}: unique scores = {rnk['score'].nunique()}")

    # Ensure one score per gene.
    # If a gene occurs multiple times, keep the occurrence with the largest score.
    rnk = (
        rnk.sort_values(["gene", "score"], ascending=[True, False], kind="mergesort")
        .drop_duplicates("gene", keep="first")
    )

    # Deterministic ranking:
    # primary sort by score, secondary sort by gene name to break ties reproducibly.
    rnk = rnk.sort_values(
        ["score", "gene"],
        ascending=[False, True],
        kind="mergesort",
    )

    # Optional: save exact GSEA input for reproducibility/debugging.
    rnk.to_csv(out_dir / f"{factor_tag}.rnk", sep="\t", index=False, header=False)

    res = gp.prerank(
        rnk=rnk,
        gene_sets=gene_set,
        outdir=str(out_dir),
        max_size=500,
        min_size=10,
        permutation_num=10000,
        seed=42,
        threads=1,
        verbose=False,
    )

    return res.res2d


def _load_gene_set_library(gene_set_name):
    try:
        lib = gp.get_library(name=gene_set_name)
    except Exception:
        lib = {}
    return lib


def _normalize_factor_name(factor, model):
    if factor in model.get_factors(df=True).columns:
        return factor
    if factor in model.get_weights(df=True).columns:
        return factor
    if factor.startswith("F") and factor[1:].isdigit():
        return f"Factor{factor[1:]}"
    return factor


def _factor_var_explained_for_genes(model, view, factor, genes):
    factors_df = model.get_factors(df=True)
    X_df = model.get_data(views=view, df=True) # we saved the feature wise zero-mean data, so no need to center here
    W_df = model.get_weights(views=view, df=True)
    factor_name = _normalize_factor_name(factor, model)
    X_df = X_df.reindex(factors_df.index)
    genes_in_data = list(set(genes) & set(X_df.columns.tolist()))
    
    X = X_df[genes_in_data].to_numpy()
    Z = factors_df[[factor_name]].to_numpy()
    W = W_df.loc[genes_in_data, [factor_name]].to_numpy()
    X_hat = Z @ W.T

    return r2_score(X.ravel(), X_hat.ravel())


gene_set_name = "Reactome_2022"
gene_set_library = _load_gene_set_library(gene_set_name)

# Run for RNA and protein (weights-based)
weights_dir = Path("../paper/gbm_mefisto/")
report_gsea_out = Path("../paper/gbm_mefisto/gbm_gsea_mefisto")

report_specs = [
    ("gene_exp", "Factor1"),
    ("gene_exp", "Factor2"),
    ("gene_exp", "Factor3"),
    ("gene_exp", "Factor4"),
]

gsea_report_results = {}
for view, factor_tag in report_specs:
    report_csv = weights_dir / f"weights_gene_exp_mefisto.csv"
    out_dir = report_gsea_out / f"{view}_{factor_tag}"
    gsea_report_results[f"{factor_tag}"] = run_gsea_from_report(
        report_csv,
        factor_tag,
        out_dir,
        gene_set=gene_set_name
    )

model_map = {
    "Z": m_Z,
    "Interpolated": m_Z_int,
    "$Z-$Interpolated": m_Z_diff,
}

summaries = []
for key, df in gsea_report_results.items():
    summaries.append(
        summarize_gsea_results(
            {key: df},
            view_label="gene_exp",
            gene_set_library=gene_set_library,
            model_map=model_map,
            top_n=20,
            view_for_r2="gene_exp",
        )
    )


example_df = next(iter(gsea_report_results.values()))
print("example terms:", example_df["Term"].head(3).tolist())
gsea_report_summary = pd.concat(summaries, ignore_index=True)
gsea_report_summary.to_csv(
    "../paper/gbm_mefisto/reactome_report_summary_top20_with_var_explained_mefisto.csv",
    index=False,
)
gsea_report_summary

In [ ]:
gsea_report_summary = pd.read_csv(
    "../paper/gbm_mefisto/reactome_report_summary_top20_with_var_explained_mefisto.csv"
)
gsea_report_summary.head()
terms = [
    "Collagen Chain Trimerization R-HSA-8948216",
    "Binding And Uptake Of Ligands By Scavenger Receptors R-HSA-2173782",
    "Collagen Chain Trimerization R-HSA-8948216",
    "Binding And Uptake Of Ligands By Scavenger Receptors R-HSA-2173782"
]

factor_order = ["Factor2", "Factor2", "Factor3", "Factor3"]

label_terms = [
    "Collagen chain trimerization",
    "Scavenger receptor uptake",
    "Collagen chain trimerization",
    "Scavenger receptor uptake",
]

pairs_df = pd.DataFrame(
    {"factor": factor_order, "term": terms, "order": range(len(terms))}
)

plot_df = (
    gsea_report_summary.merge(pairs_df, on=["factor", "term"], how="inner")[
        [
            "factor",
            "term",
            "order",
            "NES",
            "var_explained_Z",
            "var_explained_interpolated",
            "var_explained_diff",
        ]
    ]
    .drop_duplicates(subset=["factor", "term"])
.sort_values("order")
)

plot_df["row_label"] = plot_df["factor"] + " | " + plot_df["term"]
plot_df = plot_df.set_index("row_label")[
    ["NES", "var_explained_Z", "var_explained_interpolated", "var_explained_diff"]
]
plot_df[
    ["var_explained_Z", "var_explained_interpolated", "var_explained_diff"]
] = plot_df[
    ["var_explained_Z", "var_explained_interpolated", "var_explained_diff"]
] * 100  # percent

fig = plt.figure(figsize=(8.3 / 2.54, 2 / 2.54))
gs = fig.add_gridspec(1, 2, width_ratios=[0.4, 0.6], wspace=0.05)
ax_nes = fig.add_subplot(gs[0, 0])
ax = fig.add_subplot(gs[0, 1], sharey=ax_nes)

def _truncate_cmap(cmap_name, vmin=0.5, vmax=1.0, n=256):
    base = plt.get_cmap(cmap_name)
    return mcolors.LinearSegmentedColormap.from_list(
        f"{cmap_name}_trunc", base(np.linspace(vmin, vmax, n))
    )

cmap_var = _truncate_cmap("custom_cm", 0.5, 1.0)

ax = sns.heatmap(
    plot_df[[
        "var_explained_Z",
        "var_explained_interpolated",
        "var_explained_diff",
    ]],
    cmap=cmap_var,  # positive part only
    vmin=0,
    vmax=30,
    linewidths=0.25,
    linecolor="black",
    annot=True,
    fmt=".1f",
    annot_kws={"fontsize": 5},
    cbar=False,
    ax=ax,
)
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.3)
ax.set_title(r"Variance explained ($R^2$ in %)", fontsize=5, pad=3)
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_xticklabels(["Z", "Interpolated", "$Z-$Interpolated"], rotation=0, fontsize=5)
ax.tick_params(axis="both", labelsize=5, length=0, width=0.25, pad=1)
ax.tick_params(axis="y", labelleft=False)

norm = mcolors.Normalize(vmin=0, vmax=30)
sm = ScalarMappable(norm=norm, cmap=cmap_var)
sm.set_array([])

cbar = fig.colorbar(
    sm,
    ax=ax,
    fraction=0.10,
    pad=0.04,
    aspect=15,
    shrink=0.9,
)
cbar.set_ticks([0, 10, 20, 30])
cbar.ax.tick_params(labelsize=5, length=0, width=0.25, pad=1)
cbar.outline.set_visible(False)

nes_vals = plot_df[r"NES"].to_numpy()
nes_abs = np.abs(nes_vals)
y_pos = np.arange(len(nes_vals)) + 0.5
max_abs = np.nanmax(nes_abs) if len(nes_abs) else 1.0
pad = max_abs * 0.02

bar_colors = ["#9CC8FE" if val < 0 else "#FED0B4" for val in nes_vals]
ax_nes.barh(
    y_pos,
    nes_abs,
    color=bar_colors,
    height=0.8,
    edgecolor="black",
    linewidth=0.3,
)
ax_nes.set_xlim(0, 2.1)
ax_nes.set_xticks([0, 0.5, 1.0, 1.5, 2.0])
ax_nes.set_ylim(ax.get_ylim())
ax_nes.set_title("Normalized enrichment score", fontsize=5, pad=3)
ax_nes.tick_params(axis="x", labelsize=5, length=1, width=0.25, pad=1)
ax_nes.tick_params(axis="y", labelleft=True, length=0, width=0.25, pad=1)
for spine in ax_nes.spines.values():
    spine.set_linewidth(0.3)

y_labels = [textwrap.fill(label, width=45, replace_whitespace=False) for label in label_terms]
ax_nes.set_yticks(y_pos)
ax_nes.set_yticklabels(y_labels, rotation=0, fontsize=5)
ax_nes.tick_params(axis="y", pad=2)

for y, factor in zip(y_pos, factor_order):
    ax_nes.text(
        0.06,
        y,
        factor.replace("Factor", "F"),
        color="black",
        va="center",
        ha="left",
        fontsize=5,
    )

for y, val, abs_val in zip(y_pos, nes_vals, nes_abs):
    sign = "+" if val >= 0 else "-"
    ax_nes.text(
        abs_val + pad,
        y,
        sign,
        va="center",
        ha="left",
        fontsize=5,
        fontweight="bold",
    )

# Avoid tight_layout here; it can shift inset colorbar geometry and distort the tip.
fig.subplots_adjust(left=0.06, right=0.94, top=0.95, bottom=0.12, wspace=0.05)
plt.savefig(
    "../paper/gbm_mefisto/gbm_gsea_terms_var_explained_heatmap_mefisto.pdf",
    bbox_inches="tight",
    transparent=True,
)
plt.show()

In [ ]:
gsea_report_summary = pd.read_csv(
    "../paper/gbm_mefisto/reactome_report_summary_top20_with_var_explained_mefisto.csv" 
)

terms = [
    "Binding And Uptake Of Ligands By Scavenger Receptors R-HSA-2173782",
    "Collagen Chain Trimerization R-HSA-8948216",
]
table_cols = [
    "factor",
    "term",
    "NES",
    "NOM_pval",
    "FDR_qval",
    "var_explained_Z",
    "var_explained_interpolated",
    "var_explained_diff",
]

table_df = gsea_report_summary[table_cols].copy()
table_df = table_df[table_df["factor"] != "Factor4"]
table_df[[
    "var_explained_Z",
    "var_explained_interpolated",
    "var_explained_diff",
]] *= 100
table_df = table_df.sort_values(["factor", "FDR_qval", "NOM_pval"]).reset_index(drop=True)

ecm_term = "Collagen Formation R-HSA-1474290"
highlight_mask = table_df["term"].isin(terms) & (
    (table_df["term"] != ecm_term) | (table_df["factor"] == "Factor2")
)
highlight_rows = table_df.index[highlight_mask].tolist()
highlight_row_colors = {
    idx: ("#FED0B4" if table_df.loc[idx, "NES"] >= 0 else "#9CC8FE")
    for idx in highlight_rows
}

table_display = table_df.copy()
table_display["NES"] = table_display["NES"].map(lambda x: f"{x:.2f}")
table_display["NOM_pval"] = table_display["NOM_pval"].map(lambda x: f"{x:.2e}")
table_display["FDR_qval"] = table_display["FDR_qval"].map(lambda x: f"{x:.2e}")
for col in [
    "var_explained_Z",
    "var_explained_interpolated",
    "var_explained_diff",
]:
    table_display[col] = table_display[col].map(lambda x: f"{x:.1f}")

term_wrap_width = 45
table_display["term"] = table_display["term"].map(
    lambda x: textwrap.fill(str(x), width=term_wrap_width, break_long_words=False)
)
term_line_counts = table_display["term"].str.count("\n") + 1
wrapped_row_indices = table_display.index[term_line_counts > 1].tolist()

fig_table, ax_table = plt.subplots(figsize=(18 / 2.54, 20/ 2.54))
ax_table.axis("off")

tbl = ax_table.table(
    cellText=table_display.values,
    colLabels=[
        "Factor",
        "Pathway",
        "NES",
        r"NOM $\boldsymbol{p}$-val",
        r"FDR $\boldsymbol{q}$-val",
        r"$\boldsymbol{R}^\boldsymbol{2}$ $Z$ (%)",
        r"$\boldsymbol{R}^\boldsymbol{2}$ $Z_{\mathrm{interp}}$ (%)",
        r"$\boldsymbol{R}^\boldsymbol{2}$ $Z-Z_{\mathrm{interp}}$ (%)",
    ],
    colWidths=[0.065, 0.35, 0.04, 0.1, 0.09, 0.115, 0.1, 0.14],
    cellLoc="left",
    colLoc="left",
    loc="center",
    bbox=[0, 0, 1, 1],
)

if len(table_display) > 0:
    base_body_height = tbl[(1, 0)].get_height()
    for row_idx in wrapped_row_indices:
        table_row = row_idx + 1
        for col_idx in range(len(table_display.columns)):
            tbl[(table_row, col_idx)].set_height(base_body_height * 1.7)

tbl.auto_set_font_size(False)
tbl.set_fontsize(5)
for (row, col), cell in tbl.get_celld().items():
    cell.set_edgecolor("black")
    cell.set_linewidth(0.25)
    if row == 0:
        cell.set_text_props(weight="bold")
        cell.set_facecolor("#f0f0f0")
    else:
        row_idx = row - 1
        if row_idx in highlight_row_colors:
            cell.set_facecolor(highlight_row_colors[row_idx])
            if col == 1:
                cell.set_text_props(weight="bold")
    if col == 1:
        cell.PAD = 0.02

plt.savefig(
    "../paper/gbm_mefisto/gbm_gsea_report_summary_table_f2f3_mefisto.pdf",
    bbox_inches="tight",
    transparent=True,
)
plt.show()

In [ ]:
gsea_report_summary = pd.read_csv(
    "../paper/gbm_mefisto/reactome_report_summary_top20_with_var_explained_mefisto.csv" 
)

terms = [ "" ]
table_cols = [
    "factor",
    "term",
    "NES",
    "NOM_pval",
    "FDR_qval",
    "var_explained_Z",
    "var_explained_interpolated",
    "var_explained_diff",
]

table_df = gsea_report_summary[table_cols].copy()
table_df = table_df[table_df["factor"] == "Factor4"]
table_df[[
    "var_explained_Z",
    "var_explained_interpolated",
    "var_explained_diff",
]] *= 100
table_df = table_df.sort_values(["factor", "FDR_qval", "NOM_pval"]).reset_index(drop=True)

ecm_term = "Collagen Formation R-HSA-1474290"
highlight_mask = table_df["term"].isin(terms) & (
    (table_df["term"] != ecm_term) | (table_df["factor"] == "Factor2")
)
highlight_rows = table_df.index[highlight_mask].tolist()
highlight_row_colors = {
    idx: ("#FED0B4" if table_df.loc[idx, "NES"] >= 0 else "#9CC8FE")
    for idx in highlight_rows
}

table_display = table_df.copy()
table_display["NES"] = table_display["NES"].map(lambda x: f"{x:.2f}")
table_display["NOM_pval"] = table_display["NOM_pval"].map(lambda x: f"{x:.2e}")
table_display["FDR_qval"] = table_display["FDR_qval"].map(lambda x: f"{x:.2e}")
for col in [
    "var_explained_Z",
    "var_explained_interpolated",
    "var_explained_diff",
]:
    table_display[col] = table_display[col].map(lambda x: f"{x:.1f}")

term_wrap_width = 45
table_display["term"] = table_display["term"].map(
    lambda x: textwrap.fill(str(x), width=term_wrap_width, break_long_words=False)
)
term_line_counts = table_display["term"].str.count("\n") + 1
wrapped_row_indices = table_display.index[term_line_counts > 1].tolist()

fig_table, ax_table = plt.subplots(figsize=(18 / 2.54, 2/ 2.54))
ax_table.axis("off")

tbl = ax_table.table(
    cellText=table_display.values,
    colLabels=[
        "Factor",
        "Pathway",
        "NES",
        r"NOM $\boldsymbol{p}$-val",
        r"FDR $\boldsymbol{q}$-val",
        r"$\boldsymbol{R}^\boldsymbol{2}$ $Z$ (%)",
        r"$\boldsymbol{R}^\boldsymbol{2}$ $Z_{\mathrm{interp}}$ (%)",
        r"$\boldsymbol{R}^\boldsymbol{2}$ $Z-Z_{\mathrm{interp}}$ (%)",
    ],
    colWidths=[0.065, 0.35, 0.04, 0.1, 0.09, 0.115, 0.1, 0.14],
    cellLoc="left",
    colLoc="left",
    loc="center",
    bbox=[0, 0, 1, 1],
)

if len(table_display) > 0:
    base_body_height = tbl[(1, 0)].get_height()
    for row_idx in wrapped_row_indices:
        table_row = row_idx + 1
        for col_idx in range(len(table_display.columns)):
            tbl[(table_row, col_idx)].set_height(base_body_height * 1.7)

tbl.auto_set_font_size(False)
tbl.set_fontsize(5)
for (row, col), cell in tbl.get_celld().items():
    cell.set_edgecolor("black")
    cell.set_linewidth(0.25)
    if row == 0:
        cell.set_text_props(weight="bold")
        cell.set_facecolor("#f0f0f0")
    else:
        row_idx = row - 1
        if row_idx in highlight_row_colors:
            cell.set_facecolor(highlight_row_colors[row_idx])
            if col == 1:
                cell.set_text_props(weight="bold")
    if col == 1:
        cell.PAD = 0.02

plt.savefig(
    "../paper/gbm_mefisto/gbm_gsea_report_summary_table_f4_mefisto.pdf",
    bbox_inches="tight",
    transparent=True,
)
plt.show()

### Protein marker set analysis

In [ ]:

# --------------------------------------------------
# 1. Curated protein marker sets (GBM / spatial)
# --------------------------------------------------
marker_sets_protein = {

    # Proliferation
    "Cycling/proliferation (PCNA)": [
        "PCNA_protein"
    ],

    # Myeloid / Antigen-presenting cells
    "Myeloid/APC-associated": [
        "CD163_protein",
        "CD68_protein",
        "CD14_protein",
        "ITGAM_protein",
        "ITGAX_protein",
        "FCGR3A_protein",
        "HLA-DRA",
        "PTPRC_protein_1",
        "PTPRC_protein_2"
    ],

    # Lymphoid populations (T and B cells)
    "Lymphoid": [
        "CD3E_protein",
        "CD4_protein",
        "CD8A_protein",
        "CD27_protein",
        "CCR7_protein",
        "PDCD1_protein",
        "MS4A1_protein",
        "CD19_protein",
        "PAX5_protein",
        "CR2_protein",
        "CXCR5_protein",
        "CD40_protein"
    ],

    # Immune checkpoint/Survival
    "Immune checkpoint/survival": [
        "CD274_protein",
        "BCL2_protein"
    ],

    # Vascular/Perivascular
    "Vascular/perivascular": [
        "PECAM1_protein",
        "ACTA2_protein",
        "VIM_protein"
    ],

    # Structural/Tumor-associated
    "Structural/tumor-associated": [
        "EPCAM_protein",
        "KRT5_protein",
        "SDC1_protein"
    ],

    # Granulocytic
    "Granulocytic": [
        "CEACAM8_protein"
    ]
}
# --------------------------------------------------
# 2. Factor scores (Z) + data from m_Z
# --------------------------------------------------
factors_df = m_Z.get_factors(df=True).add_suffix("_Z")
factors_df_int = m_Z_int.get_factors(df=True).add_suffix("_Z_interpolated")
factors_df_diff = m_Z_diff.get_factors(df=True).add_suffix("_Z_diff")

protein_data = m_Z.get_data(views="protein", df=True)
# print(protein_data.head())

# print(protein_data.columns)

if isinstance(protein_data, pd.DataFrame):
    protein_df = protein_data.copy()
else:
    protein_df = pd.DataFrame(protein_data, index=factors_df.index)


# --------------------------------------------------
# 3. Helper functions
# --------------------------------------------------
def _compute_full_data_std(X):
    X = X - np.nanmean(X, axis=0)[None, :]
    return np.nanstd(X)


def mean_expr_per_obs(X, full_data_std):
    # if sparse.issparse(X):
    #     X = X.toarray()
    X = X - np.nanmean(X, axis=0)[None, :]
    if full_data_std != 0:
        X = X / full_data_std
    return np.nanmean(X, axis=1)


def _expand_marker_genes(df_or_index, genes):
    cols = df_or_index if isinstance(df_or_index, pd.Index) else df_or_index.columns
    out = []
    for g in genes:
        matches = [c for c in cols if c == g or c.startswith(f"{g}_protein")]
        out.extend(matches)
    return list(dict.fromkeys(out))


def score_marker_sets_expression(df, marker_sets, full_data_std, prefix="ms_"):
    """Expression-based marker-set scores (X-space)."""
    scores = {}
    present = {}
    for name, genes in marker_sets.items():
        genes_present = _expand_marker_genes(df, genes)
        if not genes_present:
            continue
        scores[f"{prefix}{name}"] = mean_expr_per_obs(
            df[genes_present].values, full_data_std
        )
        present[name] = genes_present
    scores_df = pd.DataFrame(scores, index=df.index)
    return scores_df, present


def score_marker_sets_weights(W, marker_sets):
    """Aggregate mean factor loadings (W-space)."""
    rows = []
    for name, genes in marker_sets.items():
        genes_present = _expand_marker_genes(W.index, genes)
        if not genes_present:
            continue
        rows.append(
            pd.Series(
                W.loc[genes_present].mean(axis=0),
                name=name,
            )
)
    return pd.DataFrame(rows)


def compute_marker_var_explained(model, protein_df, marker_sets, view="protein"):
    """Variance explained per factor within marker sets for a model."""
    W = model.get_weights(views=view, df=True)
    Z = model.get_factors(df=True).reindex(protein_df.index)
    var_rows = []

    for ms, genes in marker_sets.items():
        genes_present = _expand_marker_genes(protein_df, genes)
        genes_used = (
            pd.Index(genes_present)
            .intersection(W.index)
            .intersection(protein_df.columns)
)
        if len(genes_used) == 0:
            continue
        X = protein_df[genes_used].to_numpy()
        for factor in W.columns:
            if factor not in Z.columns:
                continue
            z = Z[factor].to_numpy()[:, None]
            w = W.loc[genes_used, factor].to_numpy()[None, :]
            X_hat = z @ w
            r2 = r2_score(X.ravel(), X_hat.ravel())
            var_rows.append(
                {
                    "marker_set": ms,
                    "factor": factor,
                    "var_explained": r2,
                }
)
    var_df = pd.DataFrame(var_rows)
    var_pivot = var_df.pivot(
        index="marker_set", columns="factor", values="var_explained"
)
    return var_df, var_pivot


# --------------------------------------------------
# 4. Expression-based marker scores (X)
# --------------------------------------------------
full_data_std_protein = _compute_full_data_std(protein_df.values)
scores_df, present_markers = score_marker_sets_expression(
    protein_df, marker_sets_protein, full_data_std_protein
)

score_cols = list(scores_df.columns)

# --------------------------------------------------
# 5. Correlate marker expression with factor activity (Z)
# --------------------------------------------------
expr_corr = (
    pd.concat([scores_df, factors_df], axis=1)
    .corr()
    .loc[score_cols, list(factors_df.columns)]
)

expr_corr_int = (
    pd.concat([scores_df, factors_df_int], axis=1)
    .corr()
    .loc[score_cols, list(factors_df_int.columns)]
)

expr_corr_diff = (
    pd.concat([scores_df, factors_df_diff], axis=1)
    .corr()
    .loc[score_cols, list(factors_df_diff.columns)]
)

# --------------------------------------------------
# 6. Weight-based marker scores (W)
# --------------------------------------------------
W_protein = m_Z.get_weights(views="protein", df=True)
common_features = W_protein.index.intersection(protein_df.columns)

missing_in_W = protein_df.columns.difference(W_protein.index)
missing_in_X = W_protein.index.difference(protein_df.columns)
W_protein = W_protein.loc[common_features]

marker_weight_scores = score_marker_sets_weights(W_protein, marker_sets_protein)

# Normalize weights per factor to +-1
max_abs_per_factor = marker_weight_scores.abs().max(axis=0).replace(0, np.nan)
marker_weight_scores_norm = marker_weight_scores.div(max_abs_per_factor, axis=1).fillna(0)

# --------------------------------------------------
# 7. Variance explained per factor within marker sets
# --------------------------------------------------
marker_var_explained, marker_var_explained_pivot = compute_marker_var_explained(
    m_Z, protein_df, marker_sets_protein
)
marker_var_explained_int, marker_var_explained_pivot_int = compute_marker_var_explained(
    m_Z_int, protein_df, marker_sets_protein
)
marker_var_explained_diff, marker_var_explained_pivot_diff = (
    compute_marker_var_explained(m_Z_diff, protein_df, marker_sets_protein)
)

# --------------------------------------------------
# 8. Zine X + W evidence
# --------------------------------------------------
summary_rows = []
factor_cols_W = list(W_protein.columns)

for factor in factor_cols_W:
    for ms in marker_weight_scores.index:
        summary_rows.append(
            {
                "factor": factor,
                "marker_set": ms,
                "mean_weight": marker_weight_scores.loc[ms, factor],
                "mean_weight_norm": marker_weight_scores_norm.loc[ms, factor],
                "expr_corr": (
                    expr_corr.loc[f"ms_{ms}", f"{factor}_Z"]
                    if f"ms_{ms}" in expr_corr.index
                    and f"{factor}_Z" in expr_corr.columns
                    else np.nan
                ),
                "expr_corr_interpolated": (
                    expr_corr_int.loc[f"ms_{ms}", f"{factor}_Z_interpolated"]
                    if f"ms_{ms}" in expr_corr_int.index
                    and f"{factor}_Z_interpolated" in expr_corr_int.columns
                    else np.nan
                ),
                "expr_corr_diff": (
                    expr_corr_diff.loc[f"ms_{ms}", f"{factor}_Z_diff"]
                    if f"ms_{ms}" in expr_corr_diff.index
                    and f"{factor}_Z_diff" in expr_corr_diff.columns
                    else np.nan
                ),
            }
)
Z_summary = pd.DataFrame(summary_rows)

# --------------------------------------------------
# 9. Heatmaps (Factor x marker sets, per-factor panels)
# --------------------------------------------------
expr_corr_plot = expr_corr.copy()
expr_corr_plot.columns = [c.replace("_Z", "") for c in expr_corr_plot.columns]
expr_corr_plot.index = [i.replace("ms_", "") for i in expr_corr_plot.index]

expr_corr_plot_int = expr_corr_int.copy()
expr_corr_plot_int.columns = [
    c.replace("_Z_interpolated", "") for c in expr_corr_plot_int.columns
]
expr_corr_plot_int.index = [i.replace("ms_", "") for i in expr_corr_plot_int.index]

expr_corr_plot_diff = expr_corr_diff.copy()
expr_corr_plot_diff.columns = [
    c.replace("_Z_diff", "") for c in expr_corr_plot_diff.columns
]
expr_corr_plot_diff.index = [i.replace("ms_", "") for i in expr_corr_plot_diff.index]

var_explained_plot = marker_var_explained_pivot.copy()
var_explained_plot.columns = [c.replace("_Z", "") for c in var_explained_plot.columns]

var_explained_plot_int = marker_var_explained_pivot_int.copy()
var_explained_plot_int.columns = [
    c.replace("_Z_interpolated", "") for c in var_explained_plot_int.columns
]

var_explained_plot_diff = marker_var_explained_pivot_diff.copy()
var_explained_plot_diff.columns = [
    c.replace("_Z_diff", "") for c in var_explained_plot_diff.columns
]

fig_dir = Path("../paper/gbm_mefisto")
fig_dir.mkdir(parents=True, exist_ok=True)

scores_df, present_markers = score_marker_sets_expression(
    protein_df, marker_sets_protein, full_data_std_protein
)

# Exclude empty marker sets
marker_sets_protein = {k: v for k, v in marker_sets_protein.items() if k in present_markers}
marker_set_order = list(present_markers.keys())

# Reindex all plot dataframes to the same order
expr_corr_plot = expr_corr_plot.reindex(marker_set_order)
expr_corr_plot_int = expr_corr_plot_int.reindex(marker_set_order)
expr_corr_plot_diff = expr_corr_plot_diff.reindex(marker_set_order)

var_explained_plot = var_explained_plot.reindex(marker_set_order)
var_explained_plot_int = var_explained_plot_int.reindex(marker_set_order)
var_explained_plot_diff = var_explained_plot_diff.reindex(marker_set_order)

def _factor_order_from_cols(cols):
    factors = []
    for c in cols:
        if c.startswith("Factor") and c[6:].isdigit():
            factors.append(int(c[6:]))
    if not factors:
        return list(cols)
    return [f"Factor{i}" for i in sorted(factors)]


def _factor_frame(factor, df_Z, df_interpolated, df_diff):
    return pd.DataFrame(
        {
            r"$Z$": df_Z.get(factor, pd.Series(index=df_Z.index, dtype=float)),
            r"$Z_{\mathrm{int}}$": df_interpolated.get(factor, pd.Series(index=df_Z.index, dtype=float)),
            r"$Z-Z_{\mathrm{int}}$": df_diff.get(factor, pd.Series(index=df_Z.index, dtype=float)),
        }
)


def _shifted_cmap(cmap_name, midpoint, n=256):
    """Shift cmap so 0 is at midpoint AND intensity scales with distance from 0."""
    base = plt.get_cmap(cmap_name)
    x_new = np.linspace(0, 1, n)
    
    # Map x_new through midpoint, but keep intensity proportional to distance from center
    # Below midpoint: map [0, midpoint] → [0, 0.5] linearly
    # Above midpoint: map [midpoint, 1] → [0.5, 1] linearly
    x_old = np.where(
        x_new < midpoint,
        0.5 * x_new / midpoint,
        0.5 + 0.5 * (x_new - midpoint) / (1 - midpoint),
    )
    
    # Now adjust intensity: scale saturation by distance from midpoint
    colors = base(x_old)
    
    # Calculate distance from midpoint (0 at midpoint, 1 at edges)
    dist_from_mid = np.abs(x_new - midpoint) / max(midpoint, 1 - midpoint)
    
    # Blend towards white based on how close to midpoint
    white = np.array([1, 1, 1, 1])
    for i in range(n):
        # Closer to midpoint = more white (less intense)
        blend = 1 - dist_from_mid[i]
        colors[i, :3] = colors[i, :3] * (1 - blend * 0.7) + white[:3] * blend * 0.7
    
    return mcolors.LinearSegmentedColormap.from_list(f"{cmap_name}_shifted", colors)

def _plot_factor_panels(
    dfs,
    factor_order,
    vmin,
    vmax,
    cmap,
    out_path,
    fmt=".1f",
    annot_size=5,
    tick_size=5,
    title_size=5,
    extend=None,
    annot_threshold=None,
    annot_high_color="white",
    annot_low_color="black",
    flag="corr",
    norm=None,
    cbar_ticks=None,
    cbar_ticklabels=None,
    cbar_min=None,
    cbar_max=None,
):
    n_factors = len(factor_order)
    fig, axes = plt.subplots(
        1,
        n_factors,
        figsize=(10 / 2.54, 3 / 2.54),
        gridspec_kw={"wspace": 0.05},
)
    if n_factors == 1:
        axes = [axes]

    for idx, factor in enumerate(factor_order):
        ax = axes[idx]
        data = _factor_frame(factor, dfs[r"$Z$"], dfs[r"$Z_{\mathrm{int}}$"], dfs[r"$Z-Z_{\mathrm{int}}$"])
        sns.heatmap(
            data,
            ax=ax,
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
            norm=norm,
            linewidths=0.25,
            linecolor="black",
            annot=True,
            fmt=fmt,
            annot_kws={"fontsize": annot_size},
            cbar=False,
            yticklabels=(idx == 0),
)
        if flag == "corr":
            ax.set_xticklabels([])
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(0.3)
        if annot_threshold is not None:
            values = data.to_numpy().ravel()
            for text, val in zip(ax.texts, values):
                if np.isnan(val):
                    text.set_color(annot_low_color)
                elif val > annot_threshold:
                    text.set_color(annot_high_color)
                else:
                    text.set_color(annot_low_color)
        if flag == "corr":
            ax.set_title(factor.replace("Factor", "F"), fontsize=title_size, pad=3)
        else:
            ax.set_title("")

        ax.set_xlabel("")
        ax.set_ylabel("")
        ax.tick_params(axis="x", labelsize=tick_size, length=0, width=0.25, pad=1)
        ax.tick_params(axis="y", labelsize=tick_size, length=0, width=0.25, pad=2)

    mappable = axes[0].collections[0]
    cbar = fig.colorbar(
        mappable,
        ax=axes,
        fraction=0.1,
        pad=0.02,
        extend=extend,
        aspect=20,
        shrink=0.9,
)

    if cbar_min is not None or cbar_max is not None:
        cbar.ax.set_ylim(
            cbar_min if cbar_min is not None else cbar.vmin,
            cbar_max if cbar_max is not None else cbar.vmax,
        )
    if cbar_ticks is not None:
        cbar.set_ticks(cbar_ticks)
        if cbar_ticklabels is None:
            cbar.set_ticklabels([f"{t:g}" for t in cbar_ticks])
        else:
            cbar.set_ticklabels(cbar_ticklabels)
    cbar.ax.tick_params(labelsize=5, length=0, width=0.25, pad=1)
    cbar.set_label(
        r"Expression corr." if flag == "corr" else r"Var. explained (%)",
        fontsize=5,
        labelpad=2,
)
    cbar.outline.set_visible(False)
    plt.tight_layout()
    plt.savefig(out_path, bbox_inches="tight", transparent=True)

model_labels = [r"$Z$", r"$Z_{\mathrm{int}}$",r"$Z-Z_{\mathrm{int}}$"]
factor_order = _factor_order_from_cols(expr_corr_plot.columns)

all_var_vals = pd.concat(
    [var_explained_plot, var_explained_plot_int, var_explained_plot_diff], axis=1
).to_numpy() * 100

cbar_ticks = [-10, 0, 10, 20, 30]
cbar_ticklabels = ["-10", "0", "10", "20", "30"]

_plot_factor_panels(
    {
        r"$Z$": expr_corr_plot,
        r"$Z_{\mathrm{int}}$": expr_corr_plot_int,
        r"$Z-Z_{\mathrm{int}}$": expr_corr_plot_diff,
    },
    factor_order,
    vmin=-1,
    vmax=1,
    cmap="seismic",
    out_path=fig_dir / "gbm_marker_expr_corr_Z_mefisto.pdf",
    flag="corr"
)

vmin, vmax = -10, 30
midpoint = (0 - vmin) / (vmax - vmin)  # 0 position in the colorbar
cmap_shifted = _shifted_cmap("custom_cm", midpoint)

_plot_factor_panels(
    {
        r"$Z$": var_explained_plot * 100,
        r"$Z_{\mathrm{int}}$": var_explained_plot_int * 100,
        r"$Z-Z_{\mathrm{int}}$": var_explained_plot_diff * 100,
    },
    factor_order,
    vmin=vmin,
    vmax=vmax,
    cmap=cmap_shifted,
    out_path=fig_dir / "gbm_marker_var_explained_Z_mefisto.pdf",
    annot_threshold=20,
    cbar_ticks=cbar_ticks,
    cbar_ticklabels=cbar_ticklabels,
    cbar_min=vmin,
    cbar_max=vmax,
    flag="var",
    norm=None,
)

# --------------------------------------------------
# 10. Save outputs
# --------------------------------------------------
output_dir = Path("../paper/gbm_mefisto/gbm_marker_scores_mefisto")
output_dir.mkdir(parents=True, exist_ok=True)

expr_corr.to_csv(output_dir / "protein_marker_factor_expression_correlations_mefisto.csv")

expr_corr_int.to_csv(
    output_dir / "protein_marker_factor_expression_correlations_interpolated_mefisto.csv"
)

expr_corr_diff.to_csv(
    output_dir / "protein_marker_factor_expression_correlations_diff_mefisto.csv"
)

marker_weight_scores.to_csv(output_dir / "protein_marker_factor_weights_mefisto.csv")

marker_weight_scores_norm.to_csv(
    output_dir / "protein_marker_factor_weights_normalized_mefisto.csv"
)

marker_var_explained.to_csv(
    output_dir / "protein_marker_factor_variance_explained_mefisto.csv", index=False
)

marker_var_explained_pivot.to_csv(
    output_dir / "protein_marker_factor_variance_explained_pivot_mefisto.csv"
)

marker_var_explained_int.to_csv(
    output_dir / "protein_marker_factor_variance_explained_interpolated_mefisto.csv", index=False
)

marker_var_explained_pivot_int.to_csv(
    output_dir / "protein_marker_factor_variance_explained_pivot_interpolated_mefisto.csv"
)

marker_var_explained_diff.to_csv(
    output_dir / "protein_marker_factor_variance_explained_diff_mefisto.csv", index=False
)

marker_var_explained_pivot_diff.to_csv(
    output_dir / "protein_marker_factor_variance_explained_pivot_diff_mefisto.csv"
)

Z_summary.to_csv(
    output_dir / "protein_marker_factor_Z_summary_mefisto.csv", index=False
)


In [ ]:
def _pearson_pvals(scores_df, factors_df):
    score_cols = list(scores_df.columns)
    factor_cols = list(factors_df.columns)
    pvals = pd.DataFrame(index=score_cols, columns=factor_cols, dtype=float)
    for score in score_cols:
        x = scores_df[score]
        for factor in factor_cols:
            y = factors_df[factor]
            mask = x.notna() & y.notna()
            if mask.sum() < 3:
                pvals.loc[score, factor] = np.nan
                continue
            try:
                pvals.loc[score, factor] = pearsonr(x[mask], y[mask]).pvalue
            except Exception:
                pvals.loc[score, factor] = np.nan
    return pvals

def _masked_pvals(pvals, corr, threshold=0.1):
    mask = corr.abs() > threshold
    return pvals.where(mask).stack()

expr_pvals = _pearson_pvals(scores_df, factors_df)
expr_pvals_int = _pearson_pvals(scores_df, factors_df_int)
expr_pvals_diff = _pearson_pvals(scores_df, factors_df_diff)

pvals_filtered = pd.concat(
    [
        _masked_pvals(expr_pvals, expr_corr),
        _masked_pvals(expr_pvals_int, expr_corr_int),
        _masked_pvals(expr_pvals_diff, expr_corr_diff),
    ],
    ignore_index=True,
 )

max_pval = np.nanmax(pvals_filtered.to_numpy()) if len(pvals_filtered) else np.nan
print("Max Pearson correlation p-value (|r| > 0.1):", max_pval)